# Predicting warranty claims within a year
Target: `claim_within_year`. Features are recorded at sale/registration, so all are known at prediction time. `device_serial` is an identifier and is dropped.
There is no date column and every serial appears once, so there is no time or entity structure to respect; the split is a random stratified one.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
df = pd.read_csv('data.csv')
print(df.shape, 'unique serials:', df.device_serial.nunique(), 'claim rate:', df.claim_within_year.mean())
df.describe()

## 1. Test set set aside first (20%, stratified; untouched until the final step)

In [ ]:
FEATURES = ['channel','device_age_years','price_eur','daily_hours']
train, test = train_test_split(df, test_size=0.2, stratify=df.claim_within_year, random_state=0)
X_tr, y_tr = train[FEATURES].reset_index(drop=True), train.claim_within_year.reset_index(drop=True)
X_te, y_te = test[FEATURES], test.claim_within_year
print(len(train), len(test))

## 2. Candidate models (named before any scoring)
1. `baseline`: predicts the training claim rate (trivial).
2. `logreg`: logistic regression on scaled numerics + one-hot channel.
3. `logreg_C0.1`: same, stronger regularisation.
4. `hgb_shallow`: gradient boosting, max_depth 3, 100 iterations, lr 0.05.
5. `hgb_deeper`: gradient boosting, max_depth 6, 200 iterations, lr 0.05.

Selection metric: mean CV average precision (PR-AUC; classes are imbalanced). Decision threshold: maximises F1 on pooled out-of-fold predictions of the chosen model.

In [ ]:
num = ['device_age_years','price_eur','daily_hours']
def prep(): return ColumnTransformer([('n', StandardScaler(), num), ('c', OneHotEncoder(handle_unknown='ignore'), ['channel'])])
def hgb(**k): return Pipeline([('p', prep()), ('m', HistGradientBoostingClassifier(learning_rate=0.05, random_state=0, **k))])
MODELS = {
 'baseline': lambda: Pipeline([('p', prep()), ('m', DummyClassifier(strategy='prior'))]),
 'logreg': lambda: Pipeline([('p', prep()), ('m', LogisticRegression(max_iter=1000))]),
 'logreg_C0.1': lambda: Pipeline([('p', prep()), ('m', LogisticRegression(C=0.1, max_iter=1000))]),
 'hgb_shallow': lambda: hgb(max_depth=3, max_iter=100),
 'hgb_deeper': lambda: hgb(max_depth=6, max_iter=200),
}

## 3. Cross-validation on training rows only (preprocessing is inside each pipeline, refit per fold)

In [ ]:
skf = StratifiedKFold(5, shuffle=True, random_state=1)
oof = {n: np.zeros(len(X_tr)) for n in MODELS}; rows = []
for n, mk in MODELS.items():
    ap, auc = [], []
    for a, b in skf.split(X_tr, y_tr):
        m = mk().fit(X_tr.iloc[a], y_tr.iloc[a]); p = m.predict_proba(X_tr.iloc[b])[:, 1]
        oof[n][b] = p; ap.append(average_precision_score(y_tr.iloc[b], p)); auc.append(roc_auc_score(y_tr.iloc[b], p))
    rows.append((n, np.mean(ap), np.std(ap), np.mean(auc), np.std(auc)))
cv = pd.DataFrame(rows, columns=['model','AP','AP_sd','AUC','AUC_sd']).set_index('model'); cv

In [ ]:
best = cv.AP.idxmax(); p = oof[best]
ths = np.linspace(0.02, 0.6, 117); f1s = [f1_score(y_tr, p >= t) for t in ths]
thr = float(ths[int(np.argmax(f1s))]); print('selected by CV:', best, 'threshold', round(thr, 3))

## Choice (fixed from CV only, before test scoring)
CV mean average precision: baseline 0.126, logreg 0.303, logreg_C0.1 0.304, hgb_shallow 0.294, hgb_deeper 0.255. The regularised logistic regression is marginally best, simplest, and within noise of the others (fold sd about 0.02), so I choose **logreg_C0.1** with all four features. The F1-optimal out-of-fold threshold is **0.185**.

In [ ]:
CHOSEN, THRESHOLD = 'logreg_C0.1', 0.185  # fixed from the CV results above, before touching the test set

## 4. Refit on all training rows; score the test set once (headline)

In [ ]:
final = MODELS[CHOSEN]().fit(X_tr, y_tr); pt = final.predict_proba(X_te)[:, 1]
base = MODELS['baseline']().fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
yh = pt >= THRESHOLD
print(f'TEST {CHOSEN}: AUC={roc_auc_score(y_te, pt):.3f} AP={average_precision_score(y_te, pt):.3f} '
      f'precision={precision_score(y_te, yh, zero_division=0):.3f} recall={recall_score(y_te, yh):.3f} F1={f1_score(y_te, yh):.3f}')
print(f'Baseline reference: AUC={roc_auc_score(y_te, base):.3f} AP={average_precision_score(y_te, base):.3f} (= claim rate {y_te.mean():.3f})')